In [1]:
import os
model_to_test = 'pixsplit_hgq2'
model_revision = 'Training_AdaptiveHP'
hls4ml_revision = 'VU'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, model_revision)
os.makedirs(model_dir, exist_ok=True)

description = f"""
# Model Configuration

- **Model architecture description**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Dataset**: HLS4ML LHC Jets
- **Vivado/Vitis**: 2025.2
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [2]:
import os
import keras
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
seed = 0
np.random.seed(seed)
import tensorflow as tf
tf.random.set_seed(seed)

2026-05-07 03:01:12.529062: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778115672.573198  781724 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778115672.587814  781724 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-07 03:01:12.680463: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
import numpy as np
from hgq.utils.sugar import Dataset

X_train = np.load("Data/processed_data/X_train.npy")
X_val = np.load("Data/processed_data/X_val.npy")
X_test = np.load("Data/processed_data/X_test.npy")
y_train = np.load("Data/processed_data/y_train.npy")
y_val = np.load("Data/processed_data/y_val.npy")
y_test = np.load("Data/processed_data/y_test.npy")


dataset_train = Dataset(X_train, y_train, batch_size=8192, device='gpu:0')
dataset_val = Dataset(X_val, y_val, batch_size=8192, device='gpu:0')
dataset_test = Dataset(X_test, y_test, batch_size=8192, device='gpu:0')

I0000 00:00:1778115688.596173  781724 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5272 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-05-07 03:01:30.125313: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1586057760 exceeds 10% of free system memory.
2026-05-07 03:01:36.665216: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 79302888 exceeds 10% of free system memory.
2026-05-07 03:01:36.816974: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 198256320 exceeds 10% of free system memory.
2026-05-07 03:01:37.063973: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 9912816 exceeds 10% of free system memory.
2026-05-07 03:01:37.107788: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 198256320 exceeds 10% of free system memo

In [4]:
from keras.models import Sequential
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy
from hgq.config import QuantizerConfig, QuantizerConfigScope, LayerConfigScope
from hgq.layers import QDense, QSoftmax
from hgq.constraints import MinMax

In [5]:
with (
    QuantizerConfigScope(place='all', default_q_type='kbi',overflow_mode='SAT_SYM',heterogeneous_axis=None,homogeneous_axis=(),trainable=True),
    QuantizerConfigScope(place='datalane', default_q_type='kif', overflow_mode='WRAP', f0=8, i0=4, k0=1, fc=MinMax(2, 10), ic=MinMax(2, 6), heterogeneous_axis=(), homogeneous_axis=None, trainable=True),
    LayerConfigScope(enable_ebops=True, beta0=1e-6),
   ):

        
    inputs = keras.layers.Input(shape=(60,), name='input_layer')
    x = QDense(128, activation='relu', name='dense_0')(inputs)
    x = QDense(64, activation='relu', name='dense_1')(x)
    x = QDense(32, activation='relu', name='dense_2')(x)
    x = QDense(16, activation='relu', name='dense_3')(x)
    
    outputs = QDense(3, name='dense_4')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    
loss = keras.losses.CategoricalCrossentropy(from_logits=True)
opt = keras.optimizers.Adam(learning_rate=5e-3)

model.compile(opt, loss, metrics=['accuracy'], jit_compile=True, steps_per_execution=32)

In [6]:
model.summary()

# Save the model summary to a text file
with open(os.path.join(model_dir, "summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 60)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_0 (QDense)                │ (None, 128)            │        31,238 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (QDense)                │ (None, 64)             │        33,030 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (QDense)                │ (None, 32)             │         8,326 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (QDense)                │ (None, 16)             │         2,118 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (QDense)                │ (None, 3)              │           210 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 74,922 (237.80 KB)

 Trainable params: 56,174 (219.43 KB)

 Non-trainable params: 18,748 (18.37 KB)

In [7]:
import keras
import numpy as np
from math import cos, pi
from hgq.utils.sugar import BetaScheduler, Dataset, FreeEBOPs, ParetoFront, PBar, PieceWiseSchedule
from keras.callbacks import CSVLogger, LearningRateScheduler

OUTPUT_PATH_PARETO = os.path.join(model_dir, f'model_outputs')
OUTPUT_PATH_LOG = os.path.join(model_dir, f"{str(model_to_test)}_{str(model_revision)}_log.csv")

if not os.path.exists(OUTPUT_PATH_PARETO):
    os.makedirs(OUTPUT_PATH_PARETO)
    print(f"Created new folder: {OUTPUT_PATH_PARETO}")

def cosine_decay_restarts_schedule(
    initial_learning_rate: float, first_decay_steps: int, t_mul=1.0, m_mul=1.0, alpha=0.0, alpha_steps=0
):
    def schedule(global_step):
        n_cycle = 1
        cycle_step = global_step
        cycle_len = first_decay_steps
        while cycle_step >= cycle_len:
            cycle_step -= cycle_len
            cycle_len *= t_mul
            n_cycle += 1

        cycle_t = min(cycle_step / (cycle_len - alpha_steps), 1)
        lr = alpha + 0.5 * (initial_learning_rate - alpha) * (1 + cos(pi * cycle_t)) * m_mul ** max(n_cycle - 1, 0)
        return lr

    return schedule

pbar = PBar(
        'loss: {loss:.2f}/{val_loss:.2f} - acc: {accuracy:.4f}/{val_accuracy:.4f} - lr: {learning_rate:.2e} - beta: {beta:.1e}'
    )
ebops = FreeEBOPs()
pareto = ParetoFront(
        OUTPUT_PATH_PARETO,
        ['val_accuracy', 'ebops'],
        [1, -1],
        enable_if=lambda logs: logs.get("val_accuracy", 0) > 0.70,
        fname_format='epoch={epoch}-val_acc={val_accuracy:.3f}-ebops={ebops}.keras',
    )

beta_sched = BetaScheduler(PieceWiseSchedule([(0, 1e-6, 'constant'), (50, 1e-6, 'log'), (2000, 1e-3, 'constant')]))
lr_sched = LearningRateScheduler(
        cosine_decay_restarts_schedule(5e-3, 350, t_mul=1.0, m_mul=0.94, alpha=1e-5, alpha_steps=25)
    )

csv_logger = CSVLogger(OUTPUT_PATH_LOG, append=True, separator=';')

#Fixed HP training without beta_sched and lr_sched.
#Adaptive HP training utilizes schedulers for beta and learning rate with cosine decay restarts.
callbacks = [ebops, beta_sched, lr_sched, pbar, pareto, csv_logger]

Created new folder: /home/slopin/HLS4ML_testbench_KV260/development/pixsplit/pixsplit_hgq2/Training_AdaptiveHP/model_outputs


In [8]:
model.fit(dataset_train, epochs=2000, validation_data=dataset_val,callbacks=callbacks, verbose=0)

  0%|          | 0/2000 [00:00<?, ?epoch/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1778115858.584993  791690 service.cc:148] XLA service 0x77c088006dc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778115858.585204  791690 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-05-07 03:04:18.788217: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778115859.511800  791690 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-05-07 03:04:20.658845: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2470', 3304 bytes spill stores, 3280 bytes spill loads

2026-05-07 03:04:24.136198: I external/local_xla/xla/

In [ ]:
import os, glob
from keras.models import load_model
from hgq.utils import trace_minmax

# 1. Get all Pareto models
files = glob.glob(os.path.join(OUTPUT_PATH_PARETO,'*.keras'))

# 2. Sort them by the EBOPs number in the filename
# This assumes filename: "...-ebops=123.45-..."
files.sort(key=lambda x: float(x.split('ebops=')[1].split('.')[0]))
smallest = files[0]
print(f"Done! Picked {smallest}")

model_export = load_model(smallest)
ebops = trace_minmax(model_export, X_test, verbose=False)

score = model_export.evaluate(dataset_test, verbose=0)
print("Total EBOPs:", ebops)
print("Test loss:", score[0])
print("Test accuracy:", score[1])
test_acc = score[1]

model_name = f"model_{model_revision}_acc={test_acc:.4f}_ebops={ebops:.0f}.keras"
model_export.save(os.path.join(model_dir, model_name))

Done! Picked /home/slopin/HLS4ML_testbench_KV260/development/pixsplit/pixsplit_hgq2/Training_AdaptiveHP/model_outputs/epoch=769-val_acc=0.740-ebops=3792.keras
Total EBOPs: 3640
Test loss: 0.5848285555839539
Test accuracy: 0.7448660135269165
